# Try to read tensorboard events from the directory

In [1]:
%load_ext autoreload
%autoreload 2
import polars as pl
from deeptan.utils.uni import collect_tensorboard_events

/home/wuch/miniforge3/envs/sc/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
_logs_local = "/mnt/hdd2/homext/wuch/xn2p/run/logs"
_tsb_local = collect_tensorboard_events(_logs_local)

Found 31 checkpoints.



In [3]:
_logs_hpc = "/mnt/hdd2/homext/wuch/xn2p/run/logs_hpc"
_tsb_hpc = collect_tensorboard_events(_logs_hpc)

Found 268 checkpoints.



In [4]:
_tsb = pl.concat([_tsb_local, _tsb_hpc], how="diagonal", rechunk=True)
print(_tsb.columns)

_tsb.write_csv(".collected_logs/logs_tsb.csv")
_tsb.write_parquet(".collected_logs/logs_tsb.parquet")

['ckpt_path', 'log_name', 'task', 'seed', 'data', 'test/recon_MSE', 'test/recon_RMSE', 'test/recon_MAE', 'test/recon_PCC', 'test/label_MSE', 'test/label_RMSE', 'test/label_MAE', 'test/label_PCC', 'test/loss', 'test/loss_unweighted', 'val/loss', 'val/loss_unweighted', 'val/recon_MSE', 'val/recon_RMSE', 'val/recon_MAE', 'val/recon_PCC', 'val/label_MSE', 'val/label_RMSE', 'val/label_MAE', 'val/label_PCC', 'test/label_F1_weighted', 'test/label_F1_macro', 'test/label_F1_micro', 'test/label_AUROC', 'test/label_Accuracy', 'test/label_Precision', 'test/label_Recall', 'val/label_F1_weighted', 'val/label_F1_macro', 'val/label_F1_micro', 'val/label_AUROC', 'val/label_Accuracy', 'val/label_Precision', 'val/label_Recall']


In [5]:
best_ckpts = _tsb.group_by(["data", "task", "seed"]).agg([pl.col("val/loss").min()]).join(_tsb, on=["data", "task", "seed", "val/loss"], how="left")
best_ckpts = best_ckpts.sort(by=["data", "task", "seed"])

best_ckpts.write_csv(".collected_logs/best_ckpts.csv")
best_ckpts.write_parquet(".collected_logs/best_ckpts.parquet")

### Best in seed_xx

In [6]:
best_ckpts = pl.read_parquet(".collected_logs/best_ckpts.parquet")

_seeds = [f"seed_{_seed}" for _seed in range(42, 47)]

for _seed in _seeds:
    best_ckpts_seed_xx = best_ckpts.filter(pl.col("seed") == _seed)
    best_ckpts_seed_xx.write_parquet(f".collected_logs/best_ckpts_{_seed}.parquet")
    best_ckpts_seed_xx.write_csv(f".collected_logs/best_ckpts_{_seed}.csv")

### Best in seeds

In [ ]:
# import polars as pl

In [ ]:
# best_ckpts = pl.read_parquet(".collected_logs/best_ckpts.parquet")

In [ ]:
# best_seeds = best_ckpts.group_by(["data", "task"]).agg([pl.col("val/loss").min()]).join(best_ckpts, on=["data", "task", "val/loss"], how="left").sort(by=["data", "task"])

In [ ]:
# best_seeds.write_csv(".collected_logs/best_seeds.csv")
# best_seeds.write_parquet(".collected_logs/best_seeds.parquet")